# Link Prediction in Social Networks
## Notebook 01: Exploratory Data Analysis & Graph Topology Analysis

### Project-Based Learning (PBL) Objective
Understand the structure, topology, degree distribution, and community characteristics of real-world social networks (Stanford SNAP Facebook Ego-network).

In [ ]:
import sys
sys.path.append('..')

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_graph
from src.graph_analytics import compute_graph_metrics, get_degree_distribution, compute_node_centralities

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

### 1. Load Social Graph

In [ ]:
G = load_graph(dataset_name="facebook_ego", sample_nodes=800, data_dir="../data/raw")
print(f"Graph Name: {G.name}")
print(f"Number of Nodes (|V|): {G.number_of_nodes():,}")
print(f"Number of Edges (|E|): {G.number_of_edges():,}")

### 2. Compute Global Network Metrics

In [ ]:
metrics = compute_graph_metrics(G)
metrics_df = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])
metrics_df

### 3. Degree Distribution & Scale-Free Network Analysis
Social networks typically exhibit a heavy-tailed power law degree distribution: $P(k) \sim k^{-\gamma}$.

In [ ]:
deg_info = get_degree_distribution(G)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist([d for _, d in G.degree()], bins=30, color='#38bdf8', edgecolor='black', alpha=0.8)
ax1.set_title("Degree Distribution Histogram")
ax1.set_xlabel("Degree (k)")
ax1.set_ylabel("Frequency")

# Log-Log Plot with Power Law fit
ax2.scatter(deg_info["degrees"], deg_info["probabilities"], color='#0284c7', label="Empirical P(k)")
gamma = deg_info["power_law_gamma"]
if gamma > 0:
    k_vals = np.array(deg_info["degrees"])
    c_fit = deg_info["probabilities"][0] * (k_vals[0] ** gamma)
    ax2.plot(k_vals, c_fit * (k_vals ** -gamma), 'r--', label=f"Power-Law Fit (γ={gamma:.2f})")
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_title(f"Log-Log Degree Distribution (Power Law γ = {gamma:.2f})")
ax2.set_xlabel("Degree (k) [Log scale]")
ax2.set_ylabel("P(k) [Log scale]")
ax2.legend()

plt.tight_layout()
plt.show()

### 4. Node Centralities & Top Influential Users

In [ ]:
df_cent = compute_node_centralities(G, top_k=10)
print("Top 10 Influential Nodes:")
df_cent.head(10)